<a href="https://colab.research.google.com/github/Vinicius-Jose/langchain_notebooks/blob/main/RAG_EDUCATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## RAG EDUCATION

This notebook focuses on generating concise study summaries and targeted questions with answers for a given subject. It leverages the OpenAlex API to retrieve relevant academic articles and employs Retrieval Augmented Generation (RAG) techniques to extract and utilize key information from these documents.


## Installing Dependencies
Installing dependencies from Qdrant, requests, rich, langchain, langchain_core, langchain_groq

In [1]:
!pip install accelerate aiohappyeyeballs aiohttp aiosignal altair annotated-types anthropic anyio attrs backoff bm25s certifi charset-normalizer cohere colorama colorlog colpali-engine contourpy cycler datasets==3.6.0 dill distro einops fastavro filelock fonttools frozenlist fsspec gputil grpcio grpcio-tools h11 h2 hf-transfer hpack httpcore httpx huggingface-hub hyperframe idna importlib-metadata jinja2 jiter joblib jsonschema jsonschema-specifications kiwisolver markdown-it-py markupsafe matplotlib mdurl mpmath multidict multiprocess narwhals networkx numpy openai packaging pandas pdf2image peft pillow portalocker protobuf psutil pyarrow pyarrow-hotfix pydantic pydantic-core pygments pymupdf pymupdf4llm pyparsing pystemmer python-dateutil python-dotenv pytz pyyaml qdrant-client referencing regex requests requests-mock rich rich-theme-manager rpds-py safetensors scikit-learn scipy seaborn semantic-chunkers semantic-router sentence-transformers setuptools six sniffio stamina sympy tabulate tenacity threadpoolctl tiktoken tokenizers torch tqdm transformers typing-extensions tzdata urllib3 vegafusion vegafusion-python-embed vl-convert-python xxhash yarl zipp --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.7/109.7 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which ve

In [1]:
!pip install langchain_groq langchain langgraph langchain_core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 18.0 MB/s eta 0:00:00


In [50]:
!pip install trustcall

## Creating a console for pretty print

In [65]:
from rich.console import Console
from rich.style import Style
import pathlib
from rich_theme_manager import Theme, ThemeManager

THEMES = [
    Theme(
        name="dark",
        description="Dark mode theme",
        tags=["dark"],
        styles={
            "repr.own": Style(color="#e87d3e", bold=True),      # Class names
            "repr.tag_name": "dim cyan",                        # Adjust tag names
            "repr.call": "bright_yellow",                       # Function calls and other symbols
            "repr.str": "bright_green",                         # String representation
            "repr.number": "bright_red",                        # Numbers
            "repr.none": "dim white",                           # None
            "repr.attrib_name": Style(color="#e87d3e", bold=True),    # Attribute names
            "repr.attrib_value": "bright_blue",                 # Attribute values
            "default": "bright_white on black"                  # Default text and background
        },
    ),
    Theme(
        name="light",
        description="Light mode theme",
        styles={
            "repr.own": Style(color="#22863a", bold=True),          # Class names
            "repr.tag_name": Style(color="#00bfff", bold=True),     # Adjust tag names
            "repr.call": Style(color="#ffff00", bold=True),         # Function calls and other symbols
            "repr.str": Style(color="#008080", bold=True),          # String representation
            "repr.number": Style(color="#ff6347", bold=True),       # Numbers
            "repr.none": Style(color="#808080", bold=True),         # None
            "repr.attrib_name": Style(color="#ffff00", bold=True),  # Attribute names
            "repr.attrib_value": Style(color="#008080", bold=True), # Attribute values
            "default": Style(color="#000000", bgcolor="#ffffff"),   # Default text and background
        },
    ),
]

theme_dir = pathlib.Path("themes").expanduser()
theme_dir.expanduser().mkdir(parents=True, exist_ok=True)

theme_manager = ThemeManager(theme_dir=theme_dir, themes=THEMES)


dark = theme_manager.get("dark")

from rich.console import Console

dark = theme_manager.get("dark")
# Create a console with the dark theme
console = Console(theme=dark)


In [77]:
from rich.text import Text
from rich.panel import Panel
def console_print(text, title):
  response_text = Text(text)
  styled_panel = Panel(
      response_text,
      title=title,
      expand=False,
      border_style="bold green",
      padding=(1, 1)
  )

  console.print(styled_panel)

## Downloading articles from OPENALEX based on user query

In [2]:
def search_online_articles_open_alex(subject:str):
    url = "https://api.openalex.org/works"
    params = {"search":subject,"filter":"open_access.oa_status:green"}
    response = requests.get(url, params=params)
    return response.json().get("results")

In [3]:
import os
import pymupdf4llm
import requests
from time import sleep
import stamina

def download_pdf_file(url:str, data_folder:str = "data"):
  if not os.path.exists(data_folder):
    os.makedirs(data_folder)
  headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': url,
        'DNT': '1',
        'Connection': 'keep-alive'
    }
  response = requests.get(url, headers=headers)
  local_pdf_path = os.path.join(data_folder, url.split('/')[-1])
  with open(local_pdf_path, 'wb') as f:
      f.write(response.content)

  md_text = pymupdf4llm.to_markdown(local_pdf_path, page_chunks=True)
  return md_text

## Creating a Vector database and chunking the documents to store in Qdrant database

In [4]:
from semantic_chunkers import StatisticalChunker
from semantic_router import encoders

class Chunker:
  def __init__(self,):
    self.encoder = encoders.HuggingFaceEncoder(name="sentence-transformers/all-MiniLM-L6-v2")
    self.chunker = StatisticalChunker(
    encoder=self.encoder,
    min_split_tokens=100,
    max_split_tokens=500,
    plot_chunks=False,
    enable_statistics=True,
  )
  def create_chunks(self,documents:list[str]):
    chunks = self.chunker(docs=documents)
    return chunks



In [5]:
chunker = Chunker()

def create_chunks(document: list[dict], metadata:dict = None):
  chunks = []
  for page in document:
    page_chunks = chunker.create_chunks([page["text"]])
    for chunk in page_chunks[0]:
      chunk.metadata = page["metadata"]
      if metadata:
        chunk.metadata.update(metadata)
      chunks.append(chunk)
  return chunks

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [28]:
from sentence_transformers import SentenceTransformer
import uuid
from qdrant_client import models, QdrantClient

class VectorDatabase:
  def __init__(self, collection_name:str, encoder = SentenceTransformer('all-MiniLM-L6-v2'), distance=models.Distance.COSINE ):
    self.collection_name = collection_name
    self.encoder = encoder
    self.qdrant = QdrantClient(":memory:")
    self.collection = self.qdrant.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(),
        distance=distance
    )
)

  def save_chunks(self,chunks):
    self.qdrant.upload_points(
    collection_name=self.collection_name ,
    points=[
        models.PointStruct(
            id=uuid.uuid5(uuid.NAMESPACE_URL, f"{i}-{chunk.metadata.get('title')}").hex,
            vector=self.encoder.encode([chunk.content]).tolist(),
            payload={
                "document": chunk.content,
                "metadata": chunk.metadata,
                "doc_id": i
            }
        ) for i,chunk in enumerate(chunks)

    ]
)

  def search(self,query:str, limit=2):
    hits = self.qdrant.search(
        collection_name=self.collection_name,
        query_vector=self.encoder.encode(query).tolist(),
        limit=limit
    )
    return hits

In [29]:
COLLECTION_NAME = "articles"
database = VectorDatabase(COLLECTION_NAME)

/tmp/ipython-input-2245949310.py:10: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  self.collection = self.qdrant.recreate_collection(


In [104]:
# Only the first 3 documents are downloaded

In [30]:
def retrieve_articles_save_chunks(subject:str):
  papers = search_online_articles_open_alex(subject)
  urls = [paper["open_access"]["oa_url"] for paper in papers[:3] ]
  documents = [download_pdf_file(url) for url in urls]
  documents_chunks = []
  for i,document in enumerate(documents):
    metadata = papers[i]
    metadata_filter = {"title":metadata.get("title"),"publication_year":metadata.get("publication_year"),"link":urls[i],
                       "author":",".join([author.get("raw_author_name") for author in papers[i].get("authorships")])}
    documents_chunks.append(create_chunks(document,metadata_filter))
    for chunks in documents_chunks:
      database.save_chunks(chunks)


## Generating content based on retrieve documents, and using LLM to optimize query to search documents.

Obs: You will need to create an GROQ API KEY and put in your colab secrets GROQ_API_KEY
https://console.groq.com/home

In [86]:
from langchain_core.messages import HumanMessage, SystemMessage
query_system_prompt="""You are an AI assistant specializing in optimizing natural language queries for searching academic articles. Your task is to take a user's query and transform it into a concise and effective search query that will yield relevant results from an article database. Consider using keywords and relevant phrases for the search system.
Transform the user query into an optimized search query for finding articles.
The query must be in english, if the requested query is not in english, translate first and return the optimized query in english.
The optimized query must be a single phrase matching the requirements listed above"""
def create_optimized_query(query:str):
  messages = [SystemMessage(content=query_system_prompt),HumanMessage(content=query)]
  llm_output = llm.with_structured_output(QueryArticle)
  query = llm_output.invoke(messages).optimized_query
  return query

In [87]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [110]:
def search_and_retrieve(subject:str):
  subject = create_optimized_query(subject)
  print("Optimized  : ", subject)
  hits = database.search(subject,10)
  if not hits or hits[0].score < 0.5:
    print("not found : ", subject)
    retrieve_articles_save_chunks(subject)
    return search_and_retrieve(subject)
  else:
    return hits

In [89]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="deepseek-r1-distill-llama-70b")

In [90]:
prompt = """You are an expert AI assistant specializing in creating study materials from provided text. Your goal is to help users understand complex topics by summarizing content and generating targeted questions with answers, all while maintaining clear references to the source material.
Generate a concise study text summarizing the key information from the context. This study text must be no more than 4 paragraphs long and must include the references from the context.
Following the study text, create exactly 5 distinct questions and provide their corresponding answers. These questions and answers should be directly derived from and supported by the provided context and the study text you generated.
For each question and answer pair, you MUST include the full reference information from the context that was used to construct it. This reference information should include the author, title, source and page number if available from the context.
Ensure the questions cover different aspects of the context and are suitable for studying.
Generate the study materials STRICTLY based on the provided context and structure the output according to the specified schema. Do not attempt to perform any external actions or calls.
Given the following context, which includes information about research articles, their authors, titles, pages,source,  and content:
{context}
"""

In [91]:
from pydantic import BaseModel, Field
from typing import List

class Reference(BaseModel):
  "Reference schema with type fields"
  author: str = Field(description="Author of the reference")
  title: str = Field(description="Title of the reference")
  page: str = Field(description="Page number of the reference")
  source:str = Field(description="Source of the reference")

class Question(BaseModel):
  "Question schema with type fields"
  question: str = Field(description="The question text")
  answer: str = Field(description="Answer for the question")
  reference: Reference = Field(description="Reference used to create the question")

class StudyMaterial(BaseModel):
  "StudyMaterial schema with type fields"
  summary: str  = Field(description="Summary of the context")
  questions: List[Question] = Field(description="List of questions")
  references: List[Reference] = Field(description="List of references used in Summary")

class QueryArticle(BaseModel):
  optimized_query:str = Field(description="The optimized title to search articles")

In [92]:
def create_message(subject:str):
  hits = search_and_retrieve(subject)
  context=" "
  for i,hit in enumerate(hits):
    text = hit.payload.get("document")
    metadata = hit.payload.get("metadata")
    context += f"{i}. Author {metadata.get('author')} - Title: {metadata.get('title')} - Page {metadata.get('page')} - Source:{metadata.get('link')} - Content: {text} \n"
  messages = [SystemMessage(content=prompt.format(context=context)),HumanMessage(content=subject)]
  return messages


In [101]:
def print_study_material(study_material:StudyMaterial,subject):
  console_print(study_material.summary, subject)
  for question in study_material.questions:
    console_print(question.answer +f"\n\nReference: Author:{question.reference.author} \nTitle:{question.reference.title} \nPage:{question.reference.page} \nSource: {question.reference.source}", question.question)
  for reference in study_material.references:
    console_print(f"Author:{reference.author} \nTitle:{reference.title} \nPage:{reference.page} \nSource: {reference.source}", reference.title)

In [94]:
from trustcall import create_extractor
trustcall_extractor = create_extractor(
    llm,
    tools=[StudyMaterial],
    tool_choice="StudyMaterial"
)
def generate_study_material(subject:str):
  messages = create_message(subject)
  response = trustcall_extractor.invoke({"messages":messages})
  response = response.get("responses")[0]
  print_study_material(response,subject)
  return response

In [112]:
subject = "Mary Shelley and the terror and sci-fi in Frankstein"
study_material = generate_study_material(subject)

Optimized  :  Mary Shelley Frankenstein terror horror science fiction analysis


/tmp/ipython-input-2245949310.py:36: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = self.qdrant.search(


╭───────────────────────────── Mary Shelley and the terror and sci-fi in Frankstein ──────────────────────────────╮
│                                                                                                                 │
│ Mary Shelley's *Frankenstein* is a foundational text in science fiction, exploring themes of scientific hubris, │
│ monstrosity, and the ethical implications of creation. The novel's creature, often misunderstood as a mindless  │
│ monster, is a deeply human figure who grapples with identity, rejection, and the search for belonging. This     │
│ narrative has been reinterpreted in various forms of media, including films like Vincenzo Natali's *Splice*,    │
│ which updates the Frankenstein mythos to contemporary genetic engineering. Both the novel and its modern        │
│ adaptations reflect societal anxieties about scientific progress and the consequences of playing god. The       │
│ monster, as a cultural symbol, continues to serve as a metaphor for moral struggle and the 'other,' resonating  │
│ with audiences in an age saturated with insecurity and uncertainty.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────── What is the significance of Mary Shelley's *Frankenstein* in the context of science fiction? ──────────╮
│                                                                                                                 │
│ Mary Shelley's *Frankenstein* is considered a foundational text in science fiction, as it explores themes of    │
│ scientific hubris, monstrosity, and the ethical implications of creation. The novel's creature, often           │
│ misunderstood as a mindless monster, is a deeply human figure who grapples with identity, rejection, and the    │
│ search for belonging.                                                                                           │
│                                                                                                                 │
│ Reference: Author:Lars Schmeink                                                                                 │
│ Title:Biopunk Dystopias Genetic Engineering, Society and Science Fiction                                        │
│ Page:128                                                                                                        │
│ Source:                                                                                                         │
│ https://openresearchlibrary.org/ext/api/media/8c500cb8-d07d-49e3-8ee9-0660f5e682f1/assets/external_content.pdf  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────── How does Vincenzo Natali's *Splice* update the Frankenstein mythos for a contemporary audience? ────────╮
│                                                                                                                 │
│ Vincenzo Natali's *Splice* updates the Frankenstein mythos by exploring genetic engineering and the creation of │
│ a posthuman being. The film negotiates both the science-fictional dimension of possibility and the dimension of │
│ consequence, focusing on the emotional relationship with the creature and societal commitments toward the newly │
│ created life.                                                                                                   │
│                                                                                                                 │
│ Reference: Author:Lars Schmeink                                                                                 │
│ Title:Biopunk Dystopias Genetic Engineering, Society and Science Fiction                                        │
│ Page:128                                                                                                        │
│ Source:                                                                                                         │
│ https://openresearchlibrary.org/ext/api/media/8c500cb8-d07d-49e3-8ee9-0660f5e682f1/assets/external_content.pdf  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─ How does the portrayal of the monster in *Frankenstein* differ from its portrayal in modern science fiction? ──╮
│                                                                                                                 │
│ In *Frankenstein*, the monster is a deeply human figure who grapples with identity and rejection. In modern     │
│ science fiction, the monster is often depersonalized, serving as a symbol of abjection and societal fears       │
│ rather than a nuanced character.                                                                                │
│                                                                                                                 │
│ Reference: Author:Lars Schmeink                                                                                 │
│ Title:Biopunk Dystopias Genetic Engineering, Society and Science Fiction                                        │
│ Page:129                                                                                                        │
│ Source:                                                                                                         │
│ https://openresearchlibrary.org/ext/api/media/8c500cb8-d07d-49e3-8ee9-0660f5e682f1/assets/external_content.pdf  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────── What cultural anxieties does the monster in *Frankenstein* and its adaptations reflect? ────────────╮
│                                                                                                                 │
│ The monster in *Frankenstein* and its adaptations reflects cultural anxieties about scientific progress, the    │
│ ethical implications of creation, and the fear of the 'other.' These anxieties are particularly relevant in     │
│ contemporary society, which is saturated with insecurity and uncertainty.                                       │
│                                                                                                                 │
│ Reference: Author:Lars Schmeink                                                                                 │
│ Title:Biopunk Dystopias Genetic Engineering, Society and Science Fiction                                        │
│ Page:130                                                                                                        │
│ Source:                                                                                                         │
│ https://openresearchlibrary.org/ext/api/media/8c500cb8-d07d-49e3-8ee9-0660f5e682f1/assets/external_content.pdf  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────── Biopunk Dystopias Genetic Engineering, Society and Science Fiction ───────────────────────╮
│                                                                                                                 │
│ Author:Lars Schmeink                                                                                            │
│ Title:Biopunk Dystopias Genetic Engineering, Society and Science Fiction                                        │
│ Page:128                                                                                                        │
│ Source:                                                                                                         │
│ https://openresearchlibrary.org/ext/api/media/8c500cb8-d07d-49e3-8ee9-0660f5e682f1/assets/external_content.pdf  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────── Biopunk Dystopias Genetic Engineering, Society and Science Fiction ───────────────────────╮
│                                                                                                                 │
│ Author:Lars Schmeink                                                                                            │
│ Title:Biopunk Dystopias Genetic Engineering, Society and Science Fiction                                        │
│ Page:130                                                                                                        │
│ Source:                                                                                                         │
│ https://openresearchlibrary.org/ext/api/media/8c500cb8-d07d-49e3-8ee9-0660f5e682f1/assets/external_content.pdf  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────── Biopunk Dystopias Genetic Engineering, Society and Science Fiction ───────────────────────╮
│                                                                                                                 │
│ Author:Lars Schmeink                                                                                            │
│ Title:Biopunk Dystopias Genetic Engineering, Society and Science Fiction                                        │
│ Page:129                                                                                                        │
│ Source:                                                                                                         │
│ https://openresearchlibrary.org/ext/api/media/8c500cb8-d07d-49e3-8ee9-0660f5e682f1/assets/external_content.pdf  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯